# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv 


In [3]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("Ticker")

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.
dd_shift = dd_px.groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)

dd_feat = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
    Hi_lo_range = lambda x: x['High'] - x['Low']
)

dd_feat


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [26]:
# Write your code below.
import pandas as pd
import numpy as np

df = dd_feat.compute().reset_index()

df.groupby("Ticker", group_keys=False)
df["Moving Average Returns"] = df["Returns"].rolling(10).mean()


Ticker                      Date  Adj Close      Close       High        Low       Open    Volume  Year  Close_lag_1   Returns  Hi_lo_range  Moving Average Returns
   DOV 2022-01-03 00:00:00+00:00 171.326782 178.330002 183.720001 177.289993 181.639999  786100.0  2022          NaN       NaN     6.430008                     NaN
   DOV 2022-01-04 00:00:00+00:00 174.343460 181.470001 183.000000 179.600006 179.960007  808600.0  2022   178.330002  0.017608     3.399994                     NaN
   DOV 2022-01-05 00:00:00+00:00 172.297089 179.339996 182.850006 178.979996 182.000000  758800.0  2022   181.470001 -0.011738     3.870010                     NaN
   DOV 2022-01-06 00:00:00+00:00 174.343460 181.470001 182.899994 179.839996 180.559998  752800.0  2022   179.339996  0.011877     3.059998                     NaN
   DOV 2022-01-07 00:00:00+00:00 175.736511 182.919998 184.050003 181.220001 182.429993 1057500.0  2022   181.470001  0.007990     2.830002                     NaN
   DOV 2022-01-1

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

Answer:
The moving average return could also have been calculated in Dask WITHOUT needing to convert to pandas. Since this dataset is quite large, I believe it would be more efficient to use Dask to perform calculations due to its ability to use parallel computing to perform tasks much more efficiently.


## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.